# TP1 - Parte 1: Arbol de procesos
## Que pide y como se resuelve

Cada letra del arbol tiene que ser un **proceso** del sistema operativo. Los hijos de un mismo nodo se crean de forma concurrente (se lanzan todos con `start()` antes de esperar a cualquiera con `waitFor()`), y el programa se mantiene vivo un rato (`Thread.sleep`) para poder verificar el arbol completo.

Se usan dos archivos:

- **`Main.java`**: es el proceso raiz (A). Arma el `ProcessBuilder` de su hijo (B) y lo lanza con `ProcessBuilder.start()`.
- **`ChildProcess.java`**: representa a cualquier otro nodo del arbol. Recibe su nombre por parametro (`args[0]`), y segun ese nombre (con un `switch`) sabe a que hijos tiene que crear: B crea a C y D, C crea a E, D crea a F y G, E crea a H e I. Los nodos F, G, H e I no tienen `case` propio: no crean a nadie, son las hojas del arbol.

Ambos usan `ProcessHandle.current()` para mostrar su PID y el PID de su padre (PPID), y terminan con un `Thread.sleep(10000)` para que el arbol quede visible un rato antes de finalizar.

## Paso 1 - Escribir `Main.java` (Proceso A, la raiz)

In [38]:
%%writefile Main.java
import java.io.IOException;

public class Main
{
    public static void main(String[] args)
    {
        ProcessBuilder processBuilderB = new ProcessBuilder("java", "ChildProcess.java", "B");
        processBuilderB.redirectErrorStream(true);
        processBuilderB.inheritIO();

        try
        {
            System.out.println(getInfoProcess());

            Process processB = processBuilderB.start();

            processB.waitFor();
            Thread.sleep(10000);
        }
        catch (IOException ex)
        {
            System.out.println("Error al iniciar Proceso Hijo. Mensaje de error: " + ex.getMessage());
        }
        catch (InterruptedException ex)
        {
            System.out.println("Error en realizar waitFor sobre el proceso hijo. Mensaje de error: " + ex.getMessage());
        }
    }

    public static String getInfoProcess()
    {
        ProcessHandle processHandle = ProcessHandle.current();
        long pid = processHandle.pid();
        long ppid = processHandle.parent().get().pid();
        return String.format("Proceso A | PID: %s | PPID: %s", pid, ppid);
    }
}


Overwriting Main.java


## Paso 2 - Escribir `ChildProcess.java` (el resto de los nodos)

In [39]:
%%writefile ChildProcess.java
import java.io.IOException;

public class ChildProcess
{
    private static final String PROCESS_B = "B";
    private static final String PROCESS_C = "C";
    private static final String PROCESS_D = "D";
    private static final String PROCESS_E = "E";

    public static void main(String[] args)
    {
        String processName = args[0];
        System.out.println("Proceso " + processName + " | " + getInfoProcess());
        switch (processName)
        {
            case PROCESS_B:
                execProcessB();
                break;
            case PROCESS_C:
                execProcessC();
                break;
            case PROCESS_D:
                execProcessD();
                break;
            case PROCESS_E:
                execProcessE();
                break;
            default:
                break;
        }
        try
        {
            Thread.sleep(10000);
        }
        catch (InterruptedException ex)
        {
            System.out.println("Error en sleep. Mensaje: " + ex.getMessage());
        }
    }

    public static String getInfoProcess()
    {
        ProcessHandle processHandle = ProcessHandle.current();
        long pid = processHandle.pid();
        long ppid = processHandle.parent().get().pid();
        return String.format("PID: %s | PPID: %s", pid, ppid);
    }

    public static void execProcessB()
    {
        ProcessBuilder processBuilderC = new ProcessBuilder("java", "ChildProcess.java", "C");
        processBuilderC.redirectErrorStream(true);
        processBuilderC.inheritIO();

        ProcessBuilder processBuilderD = new ProcessBuilder("java", "ChildProcess.java", "D");
        processBuilderD.redirectErrorStream(true);
        processBuilderD.inheritIO();

        try
        {
            Process processC = processBuilderC.start();
            Process processD = processBuilderD.start();

            processC.waitFor();
            processD.waitFor();
        }
        catch (IOException ex)
        {
            System.out.println("Error al iniciar Proceso Hijo. Mensaje de error: " + ex.getMessage());
        }
        catch (InterruptedException ex)
        {
            System.out.println("Error en realizar waitFor sobre el proceso hijo. Mensaje de error: " + ex.getMessage());
        }
    }

    public static void execProcessC()
    {
        ProcessBuilder processBuilderE = new ProcessBuilder("java", "ChildProcess.java", "E");
        processBuilderE.redirectErrorStream(true);
        processBuilderE.inheritIO();

        try
        {
            Process processE = processBuilderE.start();

            processE.waitFor();
        }
        catch (IOException ex)
        {
            System.out.println("Error al iniciar Proceso Hijo. Mensaje de error: " + ex.getMessage());
        }
        catch (InterruptedException ex)
        {
            System.out.println("Error en realizar waitFor sobre el proceso hijo. Mensaje de error: " + ex.getMessage());
        }
    }

    public static void execProcessD()
    {
        ProcessBuilder processBuilderF = new ProcessBuilder("java", "ChildProcess.java", "F");
        processBuilderF.redirectErrorStream(true);
        processBuilderF.inheritIO();

        ProcessBuilder processBuilderG = new ProcessBuilder("java", "ChildProcess.java", "G");
        processBuilderG.redirectErrorStream(true);
        processBuilderG.inheritIO();

        try
        {
            Process processF = processBuilderF.start();
            Process processG = processBuilderG.start();

            processF.waitFor();
            processG.waitFor();
        }
        catch (IOException ex)
        {
            System.out.println("Error al iniciar Proceso Hijo. Mensaje de error: " + ex.getMessage());
        }
        catch (InterruptedException ex)
        {
            System.out.println("Error en realizar waitFor sobre el proceso hijo. Mensaje de error: " + ex.getMessage());
        }
    }

    public static void execProcessE()
    {
        ProcessBuilder processBuilderH = new ProcessBuilder("java", "ChildProcess.java", "H");
        processBuilderH.redirectErrorStream(true);
        processBuilderH.inheritIO();

        ProcessBuilder processBuilderI = new ProcessBuilder("java", "ChildProcess.java", "I");
        processBuilderI.redirectErrorStream(true);
        processBuilderI.inheritIO();

        try
        {
            Process processH = processBuilderH.start();
            Process processI = processBuilderI.start();

            processH.waitFor();
            processI.waitFor();
        }
        catch (IOException ex)
        {
            System.out.println("Error al iniciar Proceso Hijo. Mensaje de error: " + ex.getMessage());
        }
        catch (InterruptedException ex)
        {
            System.out.println("Error en realizar waitFor sobre el proceso hijo. Mensaje de error: " + ex.getMessage());
        }
    }
}


Overwriting ChildProcess.java


## Paso 3 - Ejecutar

El arbol imprime sus 9 lineas apenas cada proceso arranca (no hace falta esperar a que termine nada para verlas), pero cada nivel nuevo tarda un poco en levantar su propia JVM. Por eso las celdas de abajo esperan unos segundos antes de mirar el log: si en algun momento ves menos de 9 lineas `Proceso X | PID... | PPID...`, no es un error, simplemente todavia no terminaron de arrancar todos los procesos - esperá unos segundos mas y volve a correr esa misma celda.

In [40]:
!nohup java Main.java > salidaJava 2>&1 &

### Primer vistazo al log (mientras el arbol termina de arrancar)

In [41]:
!sleep 15
!grep -v warning salidaJava

Proceso A | PID: 16551 | PPID: 1
Proceso B | PID: 16575 | PPID: 16551
Proceso C | PID: 16599 | PPID: 16575
Proceso D | PID: 16602 | PPID: 16575
Proceso E | PID: 16647 | PPID: 16599
Proceso F | PID: 16666 | PPID: 16602
Proceso G | PID: 16669 | PPID: 16602
Proceso H | PID: 16719 | PPID: 16647
Proceso I | PID: 16724 | PPID: 16647


### Ver el arbol real de procesos (pstree)

Esto es una foto del instante en que se ejecuta: como las hojas (F, G, H, I) solo viven 10 segundos desde que arrancan.

In [42]:
!pstree -pT $(pgrep -f "java Main.java" | head -1)

java(16551)───java(16575)─┬─java(16599)───java(16647)─┬─java(16719)
                          │                           └─java(16724)
                          └─java(16602)─┬─java(16666)
                                        └─java(16669)


### Log completo y final (los 9 procesos con su PID y PPID)

In [43]:
!sleep 20
!grep -v warning salidaJava

Proceso A | PID: 16551 | PPID: 1
Proceso B | PID: 16575 | PPID: 16551
Proceso C | PID: 16599 | PPID: 16575
Proceso D | PID: 16602 | PPID: 16575
Proceso E | PID: 16647 | PPID: 16599
Proceso F | PID: 16666 | PPID: 16602
Proceso G | PID: 16669 | PPID: 16602
Proceso H | PID: 16719 | PPID: 16647
Proceso I | PID: 16724 | PPID: 16647
